In [ ]:

import kagglehub


import os
import shutil
import sys
import subprocess


#Checks if you have the right version of python and are in the kaggle working directory.

print("Python version: ", sys.version)
print("Working directory: ", os.getcwd())

disk = shutil.disk_usage("/kaggle/working")


#Checks GPUs and CUDA version (BoltzGen requires CUDA 13)
result = subprocess.run(
    ["nvidia-smi"], capture_output = True,
    text = True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print("GPU check failed.")
    print(result.stderr)


In [ ]:
#BoltzGen needs to be installed in the kaggle environment

%pip install boltzgen

In [ ]:
%pip show boltzgen

In [ ]:
#GPU availability check

import torch
import sys
import subprocess

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

result = subprocess.run(
    ["boltzgen", "--help"],
    capture_output=True,
    text=True
)
print("\nBoltzGen CLI exit code:", result.returncode)
print(result.stdout[:2000])
print(result.stderr[:1000])

In [ ]:
#Copies the github repo of BoltzGen

%cd /kaggle/working
!git clone --depth 1 https://github.com/HannesStark/boltzgen.git

In [ ]:
from pathlib import Path

repo = Path("/kaggle/working/boltzgen")
examples = repo / "example"

# Lays out all of the examples to try
for path in sorted(examples.rglob("*.yaml"))[:40]:
    print(path.relative_to(repo))


In [ ]:

## This is a test run of a protein, takes about 1 hour on kaggle's T4 x2 GPUs

!boltzgen run /kaggle/working/boltzgen/example/vanilla_protein/1g13prot.yaml \
  --output workbench/test_run \
  --protocol protein-anything \
  --num_designs 10 \
  --budget 2
# --num_designs is the number of intermediate designs. In practice you will want between 10,000 - 60,000
# --budget is how many designs should be in the final diversity optimized set

In [ ]:
#Checks the input (target) .cif file. If it compiles correctly put the output into Chimera/ PyMOL and check if it matches the desired target.

!boltzgen check /kaggle/input/datasets/vargab12345/trcm-dataset/TRCm.yaml

In [ ]:
#The design run of 60 binding proteins (TCR mimics) binding to the pMHC. Takes about 4.5 hours on kaggle's T4 x2 GPUs.

!boltzgen run /kaggle/input/datasets/vargab12345/trcm-dataset/TRCm.yaml \
  --output workbench/real_run \
  --protocol protein-anything \
  --num_designs 60 \
  --budget 3
